# CardioSense: Model Training on Primary Indian Hospital Cohort
**Dataset**: Mendeley Data DOI: 10.17632/dzz48mvjht.1 (Doppala & Bhattacharyya, $N=1,000$)
**Architectures**: Logistic Regression ($L_2$), Random Forest (Bagging), XGBoost (Gradient Boosting)
**Validation Protocol**: Stratified 80/20 Train/Test Split & Stratified 5-Fold Cross-Validation ($R=42$)


In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

from src.config import get_dataset_config, PARAM_GRIDS, RANDOM_STATE, CV_FOLDS, MODELS_DIR
from src.data_loader import load_raw_dataset, process_dataset, split_and_save_data
from src.preprocessing import build_pipeline, create_preprocessor


## 1. Load, Clean, and Partition Indian Hospital Dataset


In [ ]:
# Execute stratified 80/20 split and persist train/test sets
train_df, test_df = split_and_save_data(dataset="india")
print(f"Training split: {train_df.shape} | Held-out test split: {test_df.shape}")

cfg = get_dataset_config("india")
X_train = train_df[cfg["feature_names"]]
y_train = train_df[cfg["target_column"]]
X_test = test_df[cfg["feature_names"]]
y_test = test_df[cfg["target_column"]]

print(f"Class distribution in test set: Positive = {sum(y_test==1)}, Negative = {sum(y_test==0)}")


## 2. Train Logistic Regression Baseline with 5-Fold Cross-Validation


In [ ]:
pipe_lr = build_pipeline(
    LogisticRegression(random_state=RANDOM_STATE),
    dataset="india",
    include_scaling=True
)

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
grid_lr = GridSearchCV(pipe_lr, PARAM_GRIDS["logistic_regression"], cv=cv, scoring="roc_auc", n_jobs=-1)
grid_lr.fit(X_train, y_train)

print(f"Logistic Regression Best CV ROC-AUC: {grid_lr.best_score_:.4f}")
print("Best Parameters:", grid_lr.best_params_)


## 3. Train Random Forest Classifier with 5-Fold Cross-Validation


In [ ]:
pipe_rf = build_pipeline(
    RandomForestClassifier(random_state=RANDOM_STATE),
    dataset="india",
    include_scaling=False
)

grid_rf = GridSearchCV(pipe_rf, PARAM_GRIDS["random_forest"], cv=cv, scoring="roc_auc", n_jobs=-1)
grid_rf.fit(X_train, y_train)

print(f"Random Forest Best CV ROC-AUC: {grid_rf.best_score_:.4f}")
print("Best Parameters:", grid_rf.best_params_)


## 4. Train XGBoost Classifier with 5-Fold Cross-Validation


In [ ]:
pipe_xgb = build_pipeline(
    XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss"),
    dataset="india",
    include_scaling=False
)

grid_xgb = GridSearchCV(pipe_xgb, PARAM_GRIDS["xgboost"], cv=cv, scoring="roc_auc", n_jobs=-1)
grid_xgb.fit(X_train, y_train)

print(f"XGBoost Best CV ROC-AUC: {grid_xgb.best_score_:.4f}")
print("Best Parameters:", grid_xgb.best_params_)


## 5. Evaluate All Three Architectures on Untouched Test Set (N=200)


In [ ]:
models = {
    "Logistic Regression": grid_lr.best_estimator_,
    "Random Forest": grid_rf.best_estimator_,
    "XGBoost": grid_xgb.best_estimator_
}

results = []
for name, model in models.items():
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Recall (Sensitivity)": recall_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "F1-Score": f1_score(y_test, preds),
        "ROC-AUC": roc_auc_score(y_test, probs),
        "False Negatives (FN)": fn,
        "False Positives (FP)": fp
    })

results_df = pd.DataFrame(results)
results_df


### Key Finding: Zero False Negatives with XGBoost
The champion XGBoost model achieved **100% Sensitivity (Recall = 1.000)** on the 200 held-out test cases,
correctly identifying all 116 cardiovascular disease patients without a single missed diagnosis (0 False Negatives).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, (name, model) in enumerate(models.items()):
    preds = model.predict(X_test)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx],
                xticklabels=["Absence (0)", "Presence (1)"],
                yticklabels=["Absence (0)", "Presence (1)"])
    axes[idx].set_title(f"{name}\nFN={cm[1,0]}, FP={cm[0,1]}", fontsize=11, fontweight="bold")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("Actual")
plt.tight_layout()
plt.show()


## 6. Persist Champion Pipeline Artifact


In [ ]:
save_dir = MODELS_DIR / "india"
save_dir.mkdir(parents=True, exist_ok=True)

champion_model = grid_xgb.best_estimator_
artifact_path = save_dir / "best_pipeline.joblib"
joblib.dump(champion_model, artifact_path)
print(f"Saved deployable champion pipeline to: {artifact_path}")
